<a href="https://colab.research.google.com/github/springboardmentor0328/Image_Recognition_System_Infosys_Internship_Oct2024/blob/Abhilash_S/Infosys_Fcae_Recognition_Abhilash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Deepface
!pip install dlib imutils

In [ ]:
# import required libraries
import sqlite3
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL
import io
import os
from deepface import DeepFace
import time

# Connect to SQLite database
conn = sqlite3.connect('face_recognition.db')
cursor = conn.cursor()

# Create table if not exists
cursor.execute('''CREATE TABLE IF NOT EXISTS embeddings (
                    name TEXT,
                    embedding BLOB
                 )''')
conn.commit()


def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

def bbox_to_bytes(bbox_array):
    bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
    iobuf = io.BytesIO()
    bbox_PIL.save(iobuf, format='png')
    bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))
    return bbox_bytes

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')


# Function to insert embedding into the database
def insert_embedding(name, embedding):
    cursor.execute("INSERT INTO embeddings (name, embedding) VALUES (?, ?)", (name, embedding.tobytes()))
    conn.commit()

# Function to retrieve all embeddings from the database
def get_embeddings():
    cursor.execute("SELECT name, embedding FROM embeddings")
    rows = cursor.fetchall()
    embeddings = []
    for row in rows:
        name, emb_blob = row
        embedding = np.frombuffer(emb_blob, dtype=np.float32)
        embeddings.append((embedding, name))
    return embeddings

# Capture images for new user registration
def capture_images_for_user(username, directions, num_images=5):
    user_folder = f'images/{username}'
    os.makedirs(user_folder, exist_ok=True)

    for direction in directions:
        captured_count = 0

        while captured_count < num_images:
            # Request the video frame
            label = f"Please look {direction}. Capturing image {captured_count + 1}/{num_images}"
            js_reply = video_frame(label, '')  # Capture video frame from the browser
            img = js_to_image(js_reply["img"])  # Convert the JavaScript video frame to an OpenCV image

            # Save the raw image without overlays
            img_path = os.path.join(user_folder, f"{direction}_{captured_count + 1}.jpg")
            cv2.imwrite(img_path, img)
            captured_count += 1

            # Draw fixed overlay to indicate direction
            bbox_array = np.zeros([img.shape[0], img.shape[1], 4], dtype=np.uint8)
            start_point = (200, 100)  # Fixed box coordinates
            end_point = (440, 380)
            color = (0, 255, 0, 255)  # Green box
            cv2.rectangle(bbox_array, start_point, end_point, color, 2)
            text_position = (start_point[0] + 10, start_point[1] + 30)
            cv2.putText(bbox_array, f"Look {direction}", text_position, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255, 255), 2)
            bbox_array[:, :, 3] = (bbox_array.max(axis=2) > 0).astype(int) * 255

            # Overlay on video feed
            bbox_bytes = bbox_to_bytes(bbox_array)
            video_frame(label, bbox_bytes)

def video_stream():
    js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
        stream.getVideoTracks()[0].stop();
        video.remove();
        div.remove();
        video = null;
        div = null;
        stream = null;
        imgElement = null;
        captureCanvas = null;
        labelElement = null;
    }

    function onAnimationFrame() {
        if (!shutdown) {
            window.requestAnimationFrame(onAnimationFrame);
        }
        if (pendingResolve) {
            var result = "";
            if (!shutdown) {
                captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
                result = captureCanvas.toDataURL('image/jpeg', 0.8)
            }
            var lp = pendingResolve;
            pendingResolve = null;
            lp(result);
        }
    }

    async function createDom() {
        if (div !== null) {
            return stream;
        }

        div = document.createElement('div');
        div.style.border = '2px solid black';
        div.style.padding = '3px';
        div.style.width = '100%';
        div.style.maxWidth = '600px';
        document.body.appendChild(div);

        const modelOut = document.createElement('div');
        modelOut.innerHTML = "<span>Status:</span>";
        labelElement = document.createElement('span');
        labelElement.innerText = 'No data';
        labelElement.style.fontWeight = 'bold';
        modelOut.appendChild(labelElement);
        div.appendChild(modelOut);

        video = document.createElement('video');
        video.style.display = 'block';
        video.width = div.clientWidth - 6;
        video.setAttribute('playsinline', '');
        video.onclick = () => { shutdown = true; };
        stream = await navigator.mediaDevices.getUserMedia(
            {video: { facingMode: "environment"}});
        div.appendChild(video);

        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute';
        imgElement.style.zIndex = 1;
        imgElement.onclick = () => { shutdown = true; };
        div.appendChild(imgElement);

        const instruction = document.createElement('div');
        instruction.innerHTML =
            '<span style="color: red; font-weight: bold;">' +'</span>';
        div.appendChild(instruction);
        instruction.onclick = () => { shutdown = true; };

        video.srcObject = stream;
        await video.play();

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640;
        captureCanvas.height = 480;
        window.requestAnimationFrame(onAnimationFrame);

        return stream;
    }

    async function stream_frame(label, imgData) {
        if (shutdown) {
            removeDom();
            shutdown = false;
            return '';
        }

        var preCreate = Date.now();
        stream = await createDom();

        var preShow = Date.now();
        if (label != "") {
            labelElement.innerHTML = label;
        }

        if (imgData != "") {
            var videoRect = video.getClientRects()[0];
            imgElement.style.top = videoRect.top + "px";
            imgElement.style.left = videoRect.left + "px";
            imgElement.style.width = videoRect.width + "px";
            imgElement.style.height = videoRect.height + "px";
            imgElement.src = imgData;
        }

        var preCapture = Date.now();
        var result = await new Promise(function(resolve, reject) {
            pendingResolve = resolve;
        });
        shutdown = false;

        return {'create': preShow - preCreate,
                'show': preCapture - preShow,
                'capture': Date.now() - preCapture,
                'img': result};
    }
    ''')
    display(js)

def video_frame(label, bbox):
    data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
    return data

# Build embeddings and save to database for a new user
def build_and_save_embeddings(dataset_path='images/'):
    for person in os.listdir(dataset_path):
        person_path = os.path.join(dataset_path, person)
        if os.path.isdir(person_path):
            for img_name in os.listdir(person_path):
                img_path = os.path.join(person_path, img_name)
                try:
                    embedding = DeepFace.represent(img_path, model_name='Facenet')[0]['embedding']
                    insert_embedding(person, np.array(embedding, dtype=np.float32))
                except Exception as e:
                    return "Error", 0

# Calculate cosine similarity
def cosine_similarity(emb1, emb2):
    return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

# Match the detected face with the database embeddings
def recognize_face(face_img, threshold=0.5):
    try:
        face_embedding = DeepFace.represent(face_img, model_name='Facenet')[0]['embedding']
        best_match = None
        best_score = -1

        # Fetch embeddings from the database
        embeddings = get_embeddings()

        for emb, name in embeddings:
            similarity = cosine_similarity(face_embedding, emb)
            if similarity > best_score:
                best_score = similarity
                best_match = name

        if best_score > threshold:
            return best_match, best_score
        else:
            return "Unknown", best_score
    except Exception as e:
        return "Error", 0

# Initialize dataset embeddings
embeddings = get_embeddings()

# Start video stream
video_stream()
label_html = 'Capturing...'
bbox = ''

# Option for user registration
register_new_user = input("Do you want to register a new user? (yes/no): ").strip().lower()

if register_new_user == "yes":
    username = input("Enter the new user's name: ").strip()
    directions = ["front", "left", "right", "up", "down"]
    capture_images_for_user(username, directions, num_images=5)

    # Build and save embeddings to the database
    build_and_save_embeddings()
else:
  build_and_save_embeddings()

# Start face recognition
while True:
    # build_and_save_embeddings()
    js_reply = video_frame(label_html, bbox)
    if not js_reply:
        break

    img = js_to_image(js_reply["img"])
    bbox_array = np.zeros([480, 640, 4], dtype=np.uint8)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Detect faces using Haar Cascade
    faces = face_cascade.detectMultiScale(gray)

    # Loop through detected faces
    for (x, y, w, h) in faces:
        face_img = img[y:y+h, x:x+w]  # Extract the face region
        name, confidence = recognize_face(face_img)  # Recognize the face

        if name == "Unknown" or confidence < 0.75:
            color = (255, 0, 0)  # Red for unknown faces
            name = "Unknown"
        else:
            color = (0, 255, 0)  # Green for recognized faces

        # Draw bounding box and label
        bbox_array = cv2.rectangle(bbox_array, (x, y), (x + w, y + h), color, 2)
        label = f'{name}: {confidence:.2f}'
        cv2.putText(bbox_array, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    # Convert bbox to RGBA and display it
    bbox_array[:, :, 3] = (bbox_array.max(axis=2) > 0).astype(int) * 255
    bbox_bytes = bbox_to_bytes(bbox_array)
    bbox = bbox_bytes

# Close the database connection when done
conn.close()
